# FLIR ? comparaci?n controlada del detector

Revisi?n acad?mica reproducible. El estado de ejecuci?n se obtiene de artefactos verificados; un piloto de infraestructura no acredita resultados cient?ficos.

## Research question

?C?mo cambian las m?tricas cuando grupos visualmente correlacionados permanecen ?ntegros entre particiones? Se estudia una asociaci?n bajo controles comunes. Los tests contienen ejemplos diferentes: no se identifica causalidad del leakage.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Image, display

root = Path('reports/detection')
meta = json.loads((root/'report_metadata.json').read_text(encoding='utf-8'))
def table(name, columns=None):
    path = root/'tables'/f'{name}.csv'
    if not path.exists() or path.stat().st_size < 3:
        display(HTML('<p><strong>Pendiente: no hay resultados finales validados para esta tabla.</strong></p>'))
        return
    data = pd.read_csv(path)
    if columns is not None:
        data = data[columns]
    display(data)
def figure(name):
    display(Image(filename=str(root/'figures'/f'{name}.png')))
display(HTML(f"<p><strong>Estado cient?fico: {meta['scientific_state']}</strong></p>"))

## Experimental controls

La ?nica variable experimental principal es la estrategia de partici?n. Pesos iniciales, modelo, device, batch, resoluci?n, epochs, optimizador, augmentations y detector seeds deben coincidir. El gate exige la matriz completa. El protocolo escrito antecede al primer entrenamiento real.

In [ ]:
display(pd.DataFrame([meta['fair_comparison']]))

## Split strategies

Historical conserva sus ocurrencias; random_content asigna contenidos exactos ?ntegros. C10 (DINOv2 / PaCMAP / DBSCAN) es el candidato visual principal y C12 (DINOv2 / t-SNE / HDBSCAN) el compromiso con menor temporalidad residual y peor balance. C01 permanece como referencia de balance, sin entrenamiento. No se eligi? ning?n candidato por m?tricas YOLO.

In [ ]:
table('candidate_audit', ['candidate_label','seed','noise_fraction','n_clusters_excluding_noise','class_deviation_pp','temporal_at1','temporal_at5','temporal_at10','split_stability_mean'])

## Residual correlation before training

?Qu? correlaci?n persiste antes de entrenar? Se muestran observaciones reales de los splits congelados. La temporalidad procede de nombres y es una heur?stica, no timestamps validados. Cero duplicados exactos no implica ausencia de toda dependencia.

In [ ]:
figure('01_split_correlation_context')
table('split_context')

## Detector configuration

YOLO11n, Ultralytics 8.3.203, PyTorch 2.8.0. Configuraci?n candidata final: 50 epochs, imgsz 640, SGD, lr0 0.01, patience 0, FP32 y deterministic=True. Batch/device se fijan tras medir el hardware. P/R: confidence 0.25 e IoU 0.5; AP: confidence m?nima 0.001. La fuente completa es configs/detection/yolo11n.yaml; no se optimizan umbrales con test.

## Compute protocol

El port?til no presupone CUDA ni GPU dedicada. Se registra CPU, RAM, build CUDA, disponibilidad y memoria de GPU cuando exista. El piloto peque?o usa 24/12/20 im?genes dentro de las particiones existentes y 2 epochs. La extrapolaci?n incluye overhead del piloto y no es una duraci?n medida de los 48 entrenamientos. Stage B autom?tico en CPU est? deshabilitado.

In [ ]:
budget = meta.get('compute_budget', {})
display(pd.DataFrame([{k:v for k,v in budget.items() if not isinstance(v, dict)}]))

## Pilot validation

?Funciona entrenamiento ? checkpoint ? test ? m?tricas ? bootstrap? El piloto peque?o debe conservar cinco clases de test, hashes, membership y controles. Sus m?tricas no se utilizan para elegir configuraci?n ni obtener conclusiones. La validaci?n hist?rica carece de Vehicles y Heavy Machinery; se conserva esta limitaci?n.

In [ ]:
table('pilot_validation')

## Overall detector metrics

Primaria: mAP@50?95. Secundarias: mAP@50, Precision y Recall macro sobre cinco clases fijas. Los paneles pendientes no contienen valores simulados; se habilitan con resultados completos y controlados.

In [ ]:
figure('02_map50_95_by_strategy')
figure('03_map50_by_strategy')
figure('04_precision_recall_by_strategy')
table('split_seed_summary')

## Per-class metrics

Se deben reportar todas las clases, con soporte e incertidumbre. Una clase ausente produce una m?trica indefinida; una clase presente sin predicciones produce cero seg?n la convenci?n preespecificada.

In [ ]:
figure('06_per_class_map')
table('run_metrics')

## Heavy Machinery

M?tricas de dominio preespecificadas: Recall y mAP@50; se conserva tambi?n Precision y mAP@50?95. El bajo soporte, particularmente en validation, limita la precisi?n y la selecci?n de checkpoints.

In [ ]:
figure('05_heavy_machinery_metrics')

## Training-seed variability

La dispersi?n entre detector seeds se calcula dentro de cada split. Se conservan mean, std muestral, median, min y max. Con una sola seed no se puede estimar esta variabilidad.

In [ ]:
figure('07_training_seed_variability')
table('training_seed_summary')

## Split-seed variability

La variaci?n de composici?n se describe entre medias de split seeds, separada de la aleatoriedad del entrenamiento. Historical solo tiene una composici?n: no se inventa una variabilidad entre particiones hist?ricas.

In [ ]:
figure('08_split_seed_variability')

## Historical vs random vs cluster-aware

Las diferencias se presentan como tama?os de efecto descriptivos, junto con CIs bootstrap por run y dispersi?n entre semillas. No se impone una direcci?n esperada ni se realizan pruebas indiscriminadas. Los conjuntos test difieren entre estrategias.

In [ ]:
table('effect_sizes')

## Metrics vs residual correlation

¿Cómo se relacionan los resultados del detector con el contexto residual congelado? Las siete asociaciones de `configs/detection/associations.yaml` se preespecificaron antes de completar la matriz controlada. La figura 09 usa PENDING/PARTIAL hasta verificar todas las celdas; ningún small pilot entra en la tabla científica. Cada punto representa un run, conservando ambas seeds y sus identidades.

Asociación ≠ causalidad: cambian los ejemplos de test, su composición y dificultad; la similitud residual no es una variable experimental aislada. No se ajustan regresiones ni se calculan p-values o correlaciones como conclusión. Las demás selecciones de la aplicación son EXPLORATORY VIEW.

In [ ]:
figure('09_metrics_vs_residual_similarity')
table('prespecified_associations')
table('detector_residual_associations')
table('experiment_matrix')

## Limitations

El bootstrap por imagen no corrige la dependencia residual entre frames. Las inferencias temporales no son tiempos verificados. Se preservan conflictos originales de anotaci?n. La validaci?n hist?rica no cubre dos clases. C12 tiene peor balance. El piloto peque?o selecciona cobertura deliberadamente y no caracteriza generalizaci?n. La disponibilidad de hardware y tiempo limita qu? partes se han ejecutado.

## Interpretation

Una ca?da, ausencia de cambio o aumento de mAP admite explicaciones de composici?n, dificultad, balance e incertidumbre. Ninguno demuestra por s? solo leakage ni su ausencia. Mientras la matriz final siga pendiente, no hay evidencia experimental del detector para decidir entre esas posibilidades.

## Reproducibility

Plan, freeze, matriz, vistas, checkpoints, estad?sticas por imagen, m?tricas, bootstrap y recibos permanecen locales e ignorados por Git. Los originales se leen sin modificaci?n. El c?digo, protocolo y notebook fuente s? se versionan. Consultar docs/runbooks/detection.md para reconstrucci?n y resume; los IDs excluyen fechas y rutas absolutas.

## Conclusions for the thesis

La conclusi?n debe limitarse a lo observado. La infraestructura y un piloto t?cnico pueden estar validados mientras Stage A/B siguen pendientes. La comparaci?n cient?fica requiere completar los entrenamientos controlados, revisar incertidumbre y contrastar m?tricas con la correlaci?n residual.

In [ ]:
display(HTML(f"<p><strong>{meta['scientific_state']}</strong>: {meta['fair_comparison']['completed_runs']} / {meta['fair_comparison']['expected_runs']} runs finales verificados; {meta['small_pilot_count']} pilotos peque?os registrados.</p>"))